In [ ]:
!pip install requests beautifulsoup4 python-docx
import re
import requests
import time
from bs4 import BeautifulSoup
from docx import Document

In [ ]:
url = "https://dbu.dk/resultater/hold/194942_473829/stilling"

In [ ]:
headers = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"
}

# https://whatismyip.io/http-headers
# insert user-agent info

In [ ]:
response = requests.get(url, headers=headers)

In [ ]:
response.status_code

In [ ]:
soup = BeautifulSoup(response.content, "html.parser")

In [ ]:
table_html = soup.find('table', class_='sr--pool-position--table')
rows = table_html.find('tbody').find_all('tr')

In [ ]:
# 3. Process Data
pos_v, name_v, k_v, v_v, u_v, t_v, gd_v, p_v = [], [], [], [], [], [], [], []

for row in rows:
    cells = row.find_all('td')
    if len(cells) >= 8:
        pos_v.append(cells[0].get_text(strip=True))

        name = cells[1].find('span').get_text(strip=True) if cells[1].find('span') else cells[1].get_text(strip=True)
        name_v.append(name)

        k_v.append(cells[2].get_text(strip=True))
        v_v.append(cells[3].get_text(strip=True))
        u_v.append(cells[4].get_text(strip=True))
        t_v.append(cells[5].get_text(strip=True))

        # Calculate Goal Difference with + or -
        score_text = cells[6].get_text(" ", strip=True)
        try:
            parts = score_text.split("-")
            diff = int(parts[0].strip()) - int(parts[1].strip())
            # This adds the + or - automatically
            gd_v.append(f"{diff:+d}" if diff != 0 else "0")
        except:
            gd_v.append("0")

        p_v.append(cells[7].get_text(strip=True))

# 4. Create Word Doc
doc = Document()
doc.add_heading('DBU League Data Export', 0)

def add_list_section(title, data_list):
    doc.add_heading(title, level=1)
    doc.add_paragraph("\n".join(data_list))

add_list_section('POSITIONS', pos_v)
add_list_section('CLUB NAMES', name_v)
add_list_section('K (PLAYED)', k_v)
add_list_section('V (WON)', v_v)
add_list_section('U (DRAW)', u_v)
add_list_section('T (LOST)', t_v)
add_list_section('GD (GOAL DIFFERENCE)', gd_v)
add_list_section('P (POINTS)', p_v)

# 5. Export
file_name = "DBU_Scraping.docx"
doc.save(file_name)

print("Export complete! Your vertical lists are ready.")